In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain.tools import tool
from typing import Dict, Any
from tavily import TavilyClient
from langchain_community.utilities import SQLDatabase

tavily_client = TavilyClient()

db = SQLDatabase.from_uri("sqlite:///resources/Chinook.db")


@tool
def web_search(query: str) -> Dict[str, Any]:

    """Search the web for information"""

    return tavily_client.search(query)

@tool
def sql_query(query: str) -> str:

    """Obtain information from the database using SQL queries"""

    try:
        return db.run(query)
    except Exception as e:
        return f"Error: {e}"

In [3]:
from dataclasses import dataclass

@dataclass
class UserRole:
    user_role: str = "external"

In [4]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from typing import Callable

@wrap_model_call
def dynamic_tool_call(request: ModelRequest, 
handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:

    """Dynamically call tools based on the runtime context"""

    user_role = request.runtime.context.user_role
    
    if user_role == "internal":
        pass # internal users get access to all tools
    else:
        tools = [web_search] # external users only get access to web search
        request = request.override(tools=tools) 

    return handler(request)

In [5]:
from langchain.agents import create_agent

agent = create_agent(
    model="gpt-5-nano",
    tools=[web_search, sql_query],
    middleware=[dynamic_tool_call],
    context_schema=UserRole
)

In [6]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="How many artists are in the database?")]},
    context={"user_role": "external"}
)

print(response["messages"][-1].content)

I don’t have access to your database, so I can’t see the exact number. Which database/system are you using? (SQL, MongoDB, a CSV/Sheets file, or an internal app?)

If you tell me the platform, I can give you the exact query. In the meantime, here are quick options:

- SQL (e.g., PostgreSQL/MySQL)
  - Count total rows: SELECT COUNT(*) AS artist_count FROM artists;
  - Count unique artists by id (avoids duplicates): SELECT COUNT(DISTINCT id) AS artist_count FROM artists;

- SQL with guarantee of uniqueness (if there may be duplicates in id column)
  - SELECT COUNT(*) - COUNT(DISTINCT id) AS duplicate_rows FROM artists;

- MongoDB
  - Total documents: db.artists.countDocuments({});
  - Total unique by id field: db.artists.distinct("id").length;

- CSV/Excel
  - Count rows in the artists column (e.g., IDs): in Excel, use COUNTA on the ID column.
  - Count unique IDs in Excel: use a pivot table or a formula like =SUM(1/COUNTIF(A:A, A:A)) (entered as an array formula).

If you share your sch